# 05 · Captura web: preguntarle al sitio qué páginas tiene

El paso `web` tiene dos fases. Primero se lee el `robots.txt` y el sitemap
para saber qué páginas existen de verdad; después se piden solo esas. Antes se
adivinaban las URL, y eso gastaba peticiones en páginas inexistentes.

> **Una sola fuente de verdad.** Las funciones de este notebook viven en
> `src/etl/`, se prueban con pytest y son las mismas que corre `etl` en la
> terminal y en CI. Aquí se las llama y se mira el resultado: así el notebook
> no se desactualiza cuando se corrige el código.

In [1]:
# El paquete `etl` se instala con `uv sync` en etl/.venv. Si el kernel del
# notebook no es ese entorno, se agrega src/ a la ruta como respaldo.
import sys
from pathlib import Path

ETL = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
try:
    import etl  # noqa: F401
except ModuleNotFoundError:
    sys.path.insert(0, str(ETL / "src"))

import json

import pandas as pd

from etl import config
from etl.common import configurar_registro

configurar_registro()
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 180)

ANIO = config.ANIO_POR_DEFECTO
ANIO

2025

## 1. Las reglas de recolección, en el código

No son un comentario: son atributos de la clase y constantes de `recoleccion`.

In [ ]:
from etl import crawling
from etl.pipeline import web

print(f"User-Agent             {crawling.USER_AGENT}")
print(f"respeta robots.txt     {crawling.Etico.robots_txt_obey}")
print("suplanta navegador     no (impersonate=None, stealthy_headers=False)")
print(f"pausa inicial          {crawling.SEGUNDOS_ENTRE_PETICIONES} s, adaptativa hasta {crawling.RETRASO_MAXIMO} s")
print(f"peticiones por dominio {crawling.PETICIONES_SIMULTANEAS_POR_DOMINIO} a la vez")
print(f"reintentos si bloquea  {crawling.Etico.max_blocked_retries}")
print(f"códigos que detienen   {crawling.BLOQUEO} y {crawling.SIN_ENTRADA} por dominio")
print(f"páginas extra por web  {web.PAGINAS_EXTRA_POR_EMPRESA}")
print()
print("Las reglas viven una sola vez, en crawling.Etico, y las heredan los")
print(f"dos recolectores: {web.ExploradorDeSitio.__name__} y {web.CapturaWeb.__name__}.")

## 2. Fase 1: del sitemap a las páginas que importan

`robots.txt` suele declarar dónde está el sitemap. Del sitemap se eligen las
páginas institucionales, una por tema, para que una empresa no gaste sus tres
peticiones en tres páginas de productos.

In [3]:
from etl import site

ejemplo = """<urlset>
  <url><loc>https://ejemplo.com/</loc></url>
  <url><loc>https://ejemplo.com/nosotros</loc></url>
  <url><loc>https://ejemplo.com/productos</loc></url>
  <url><loc>https://ejemplo.com/servicios/consultoria</loc></url>
  <url><loc>https://ejemplo.com/contacto</loc></url>
  <url><loc>https://ejemplo.com/blog/2025/una-noticia</loc></url>
  <url><loc>https://ejemplo.com/politica-de-privacidad</loc></url>
</urlset>"""

paginas = site.locs_de_sitemap(ejemplo)
display(pd.DataFrame({"url": paginas,
                      "tipo": [site.tipo_pagina(u) for u in paginas],
                      "puntaje": [site.puntuar_pagina(u) for u in paginas]}))
site.elegir_paginas(paginas, web.PAGINAS_EXTRA_POR_EMPRESA, "ejemplo.com")

,url,tipo,puntaje
0,https://ejemplo.com/,portada,0
1,https://ejemplo.com/nosotros,nosotros,99
2,https://ejemplo.com/productos,servicios,79
3,https://ejemplo.com/servicios/consultoria,servicios,78
4,https://ejemplo.com/contacto,contacto,59
5,https://ejemplo.com/blog/2025/una-noticia,otra,0
6,https://ejemplo.com/politica-de-privacidad,otra,0


[('https://ejemplo.com/nosotros', 'nosotros'),
 ('https://ejemplo.com/productos', 'servicios'),
 ('https://ejemplo.com/contacto', 'contacto')]

**Lectura.** Puntaje cero significa «no se visita»: el blog, la política de
privacidad y la portada, que se pide aparte. Entre dos páginas del mismo tema
gana la más cercana a la raíz.

## 3. Correr el paso

Fase 1 explora, fase 2 captura. El HTML se guarda tal cual en
`crudo/web/AAAA/`: mejorar el extractor después no obliga a volver a visitar
ninguna web.

In [4]:
manifiesto = await web.ejecutar(ANIO, limite=3)
print(f"{len(manifiesto)} páginas en el manifiesto")
manifiesto.tail(5)[["expediente", "url", "tipo_pagina", "status"]]

06:08:46  INFO    No hay webs pendientes de capturar


18 páginas en el manifiesto


,expediente,url,tipo_pagina,status
13,724114,https://fertisa.com/servicios/,servicios,200
14,72951,https://www.telconet.ec/nosotros/,nosotros,200
15,72951,https://telconet.ec/,portada,200
16,72951,https://www.telconet.ec/soluciones/,servicios,200
17,72951,https://www.telconet.ec/contactenos/,contacto,200


## 4. El plan: qué se le pidió a cada sitio y por qué

Queda guardado en plata, así que se puede auditar la recolección sin repetirla.

In [5]:
plan = pd.read_csv(config.ruta_plata(ANIO) / "plan_captura.csv")
display(plan["origen"].value_counts().to_frame("páginas"))
plan[["expediente", "url", "tipo_previsto", "es_extra", "seguir_enlaces", "origen"]].head(15)

,páginas
origen,
sitemap,4


,expediente,url,tipo_previsto,es_extra,seguir_enlaces,origen
0,72951,https://telconet.ec/,portada,False,False,sitemap
1,72951,https://www.telconet.ec/nosotros/,nosotros,True,False,sitemap
2,72951,https://www.telconet.ec/soluciones/,servicios,True,False,sitemap
3,72951,https://www.telconet.ec/contactenos/,contacto,True,False,sitemap


## 5. Qué se obtuvo de cada empresa

In [6]:
# `lista` normaliza las celdas vacías: en un DataFrame vienen como NaN, que es
# un float verdadero, así que `len(x or [])` fallaría.
from etl.pipeline.profile import lista

capturadas = manifiesto[(manifiesto["status"] == 200) & (manifiesto["sha256"].str.len() == 64)]
resumen = capturadas.groupby("expediente").agg(
    paginas=("url", "count"),
    tipos=("tipo_pagina", lambda s: ", ".join(sorted(set(s)))),
    correos=("correos", lambda s: sum(len(lista(x)) for x in s)),
    redes=("redes", lambda s: sum(len(lista(x)) for x in s)),
    pdf=("pdf_candidatos", lambda s: sum(len(lista(x)) for x in s)),
)
resumen

,paginas,tipos,correos,redes,pdf
expediente,,,,,
384,4,"nosotros, otra, portada, servicios",0,0,0
7154,3,"nosotros, portada",0,0,1
72951,4,"contacto, nosotros, portada, servicios",15,24,4
124193,3,"otra, portada, servicios",0,0,0
724114,4,"nosotros, otra, portada, servicios",0,0,0


## 6. Páginas omitidas

Una página puede pedir no ser archivada con `<meta name="robots">` aunque
`robots.txt` permita leerla. Se registra la omisión y no se guarda el HTML.

In [7]:
if "omitida" in manifiesto:
    display(manifiesto["omitida"].fillna("").replace("", "capturada").value_counts().to_frame("páginas"))
    display(manifiesto[manifiesto["omitida"].fillna("") != ""][["expediente", "url", "status", "omitida"]])
else:
    print("Ninguna página omitida todavía.")

,páginas
omitida,
capturada,18


,expediente,url,status,omitida
